In [1]:
import importlib
import BM25_and_embeddings
importlib.reload(BM25_and_embeddings)
from BM25_and_embeddings import chunks_from_markdown_files, embedding_score, bm25_score


chunks = chunks_from_markdown_files()
query = "How to generate an AWR report in HTML format ?"
bm25_score_dict = bm25_score(query, chunks)
embedding_score_dict = embedding_score(query, chunks)


resource module not available on Windows
c:\Users\yjaquier\.conda\envs\ollama\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2441.28it/s]


In [2]:
bm25_score_ranked = sorted(bm25_score_dict, key=lambda x: x.get("bm25_score", 0.0), reverse=True)
for item in bm25_score_ranked[:20]:
  print(item)

print("---------------------------------------------------------------------------------\n")
embedding_score_ranked = sorted(embedding_score_dict, key=lambda x: x.get("embedding_score", 0.0), reverse=True)
for item in embedding_score_ranked[:20]:
  print(item)

{'id': 4064, 'file': 'skills-main\\db\\sqlcl\\sqlcl-awr.md', 'chunk': '### diagnose a recent performance problem\n\n```sql\n-- list snapshots to find the relevant time window\nawr list snapshots\n\n-- generate an html report for that window (e.g., snapshots 142 to 145)\nawr create html 142 145\n```\n\nopen the resulting `.html` file in a browser for the full formatted report including top sql, wait events, load profile, and instance efficiency metrics.', 'bm25_score': 9.613201141357422}
{'id': 4071, 'file': 'skills-main\\db\\sqlcl\\sqlcl-awr.md', 'chunk': '## best practices\n\n- run `awr list snapshots` before generating a report to confirm the snapshot ids exist and span the time window you want.\n- use html format for human review; use text format when the output needs to be parsed programmatically or sent in plain-text email.\n- keep snapshot intervals short (15–30 minutes) during active problem investigation so you can isolate the exact period when performance degraded.\n- the defa

In [11]:
bm25_rank_by_id = {item["id"]: rank for rank, item in enumerate(bm25_score_ranked, start=1)}
emb_rank_by_id = {item["id"]: rank for rank, item in enumerate(embedding_score_ranked, start=1)}

# print(bm25_rank_by_id)
# print(emb_rank_by_id)

bm25_by_id = {item["id"]: item for item in bm25_score_dict}
emb_by_id = {item["id"]: item for item in embedding_score_dict}

all_ids = set(bm25_rank_by_id) | set(emb_rank_by_id)

# print(bm25_by_id)
# print(emb_by_id)
print(all_ids)

{0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221,

In [2]:
import importlib
import BM25_and_embeddings
importlib.reload(BM25_and_embeddings)
from BM25_and_embeddings import reciprocal_rank_fusion, embedding_score, bm25_score, chunks_from_markdown_files, compute_embedding_from_chunks, bm25_retriever_from_chunks

chunks = chunks_from_markdown_files()
query = "How to generate an AWR report in HTML format ?"
bm25_retriever = bm25_retriever_from_chunks(chunks)
bm25_score_dict = bm25_score(query, chunks, bm25_retriever, 100)
chunk_embeddings = compute_embedding_from_chunks(chunks)
embedding_score_dict = embedding_score(query, chunk_embeddings, chunks)

rrf_results = reciprocal_rank_fusion(bm25_score_dict, embedding_score_dict, k = 60)

for item in rrf_results[:10]:
  print(item)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3393.69it/s]


{'id': 2553, 'source': 'skills-main\\db\\performance\\awr-reports.md', 'section_id': 9, 'chunk_id': 0, 'header_path': ['AWR Reports — Automatic Workload Repository', 'Generating AWR Reports', 'HTML Report (preferred for readability)'], 'chunk': '```sql\nSELECT output\nFROM   TABLE(\nDBMS_WORKLOAD_REPOSITORY.AWR_REPORT_HTML(\nl_dbid      => (SELECT dbid FROM v$database),\nl_inst_num  => 1,\nl_bid       => 200,\nl_eid       => 201\n)\n);\n```', 'bm25_score': 8.309660911560059, 'embedding_score': 0.7179502844810486, 'rrf_score': 0.032266458495966696}
{'id': 3273, 'source': 'skills-main\\db\\sqlcl\\sqlcl-awr.md', 'section_id': 5, 'chunk_id': 0, 'header_path': ['SQLcl AWR Command', 'Common Workflows', 'Diagnose a recent performance problem'], 'chunk': '```sql\n-- List snapshots to find the relevant time window\nawr list snapshots\n\n-- Generate an HTML report for that window (e.g., snapshots 142 to 145)\nawr create html 142 145\n```  \nOpen the resulting `.html` file in a browser for the fu

In [3]:
import importlib
import BM25_and_embeddings
importlib.reload(BM25_and_embeddings)
from BM25_and_embeddings import reciprocal_rank_fusion, embedding_score, bm25_score, chunks_from_markdown_files, compute_embedding_from_chunks

query = "How to generate an AWR report in text format ?"
bm25_score_dict = bm25_score(query, chunks, bm25_retriever, 100)
embedding_score_dict = embedding_score(query, chunk_embeddings, chunks)

rrf_results = reciprocal_rank_fusion(bm25_score_dict, embedding_score_dict, k = 60)

for item in rrf_results[:10]:
  print(item)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4077.79it/s]


{'id': 3276, 'source': 'skills-main\\db\\sqlcl\\sqlcl-awr.md', 'section_id': 8, 'chunk_id': 0, 'header_path': ['SQLcl AWR Command', 'Common Workflows', 'Generate a plain-text report for scripted parsing'], 'chunk': '```sql\nawr create text 142 145\n```  \nText format is useful for automated parsing, email delivery, or logging to a file with SPOOL:  \n```sql\nSPOOL /var/reports/awr_latest.txt\nawr create text\nSPOOL OFF\n```  \n---', 'bm25_score': 10.104742050170898, 'embedding_score': 0.7155362367630005, 'rrf_score': 0.03278688524590164}
{'id': 2552, 'source': 'skills-main\\db\\performance\\awr-reports.md', 'section_id': 8, 'chunk_id': 0, 'header_path': ['AWR Reports — Automatic Workload Repository', 'Generating AWR Reports', 'Text Report (SQL*Plus)'], 'chunk': '```sql\n-- Interactive: prompts for snap IDs and instance\n@$ORACLE_HOME/rdbms/admin/awrrpt.sql\n\n-- Non-interactive using the PL/SQL function directly\nSELECT output\nFROM   TABLE(\nDBMS_WORKLOAD_REPOSITORY.AWR_REPORT_TEXT(\n